# The Anatomy of AI · Lecture 3 · Many Neurons
### The notebook

One neuron draws one straight line, and Lecture 1 ended on a diary no straight line can split. Here you build a tiny
network of three neurons that can, work out **backpropagation** by hand on one day, and then train hundreds of networks
from random starts, exactly as the lecture did. Every number is checked against the lecture's.

Each section names the lecture plate it rebuilds. A `check(...)` line prints ✓ when your number matches, and stops the
notebook if it does not. Five **Your turn** exercises sit along the way; solutions are at the end.

It runs on the CPU in about seven minutes (on Modal's CPU; free Colab is similar). Two cells train 1,200 small networks
each and take about three minutes apiece. No GPU, no downloads, no keys.

In [ ]:
# Pinned to the versions free Colab ships (Python 3.12); nothing else is needed, and no keys.
%pip install -q numpy==2.0.2 matplotlib==3.10.0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
})
INKMAP = LinearSegmentedColormap.from_list("ink", [PAPER, INK])   # a value as ink density, as on the plates


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "−")


def check(name, value, expected, tol=1e-9):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value) + 0.0:.6g}"          # + 0.0 prints −0 as 0
    else:
        ok = value == expected
        shown = repr(value)
    want = f"{expected:.6g}" if isinstance(expected, float) else expected
    print(f"{'✓' if ok else '✗'} {name}: {shown}   (lecture: {want})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

## The band diary, from Lecture 1

Lecture 1's last plate: you enjoy a walk only when it is **neither too cold nor too hot** (and not too wet). The enjoyed
days sit in a band. 42 days, drawn with Python's random generator, seed 5: 16 enjoyed, 26 not.

In [ ]:
import random

def make_diary(rule, seed, good, bad):
    rng = random.Random(seed)
    days, need = [], {True: good, False: bad}
    while need[True] or need[False]:
        w, r = rng.uniform(0.03, 0.97), rng.uniform(0.03, 0.97)
        label = rule(w, r)
        if label is None or not need[label]:
            continue
        need[label] -= 1
        days.append((w, r, label))
    return days

def band_rule(w, r):
    inside = 0.3 < w < 0.68 and r < 0.55
    edge = min(abs(w - 0.3), abs(w - 0.68)) < 0.05 or abs(r - 0.55) < 0.05
    return None if edge else inside

band = make_diary(band_rule, seed=5, good=16, bad=26)
X = np.array([(w, r) for w, r, _ in band])
Y = np.array([1.0 if e else 0.0 for *_, e in band])
check("days", len(X), 42)
check("enjoyed", int(Y.sum()), 16)

def draw_days(ax, wrong=None):
    for (w, r), y in zip(X, Y):
        ax.plot(w, r, "o", ms=6, mfc=INK if y else PAPER, mec=INK, mew=1.1)
    for i in (wrong if wrong is not None else []):
        ax.plot(*X[i], "o", ms=14, mfc="none", mec=INK, mew=0.9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")

## Plate I · The limit of one

Try every straight line: 7,200 directions, and for each every position between two days. The best gets **35 of 42**.
One neuron draws one line, so one neuron can't do it, however it learns.

In [ ]:
best, best_line = 0, None
for th in np.linspace(0, 2 * np.pi, 7200, endpoint=False):
    n = np.array([np.cos(th), np.sin(th)])
    s = X @ n
    cs = np.sort(s)
    for c in np.r_[cs[0] - 1, (cs[1:] + cs[:-1]) / 2, cs[-1] + 1]:
        right = int(np.sum((s > c) == (Y > 0)))
        if right > best:
            best, best_line = right, (n, c)
check("the best any straight line can do", best, 35)

fig, ax = plt.subplots(figsize=(4.2, 4.2))
g = np.linspace(0, 1, 201); Wg, Rg = np.meshgrid(g, g)
S = best_line[0][0] * Wg + best_line[0][1] * Rg - best_line[1]
ax.contourf(Wg, Rg, S, levels=[0, S.max()], colors="none", hatches=["////"])
ax.contour(Wg, Rg, S, levels=[0], colors=INK, linewidths=2)
draw_days(ax, wrong=[i for i in range(42) if ((X[i] @ best_line[0]) > best_line[1]) != (Y[i] > 0)])
ax.set_title("the best line: 35 of 42 (wrong days ringed)", fontsize=10); ax.set_xlabel("warmth"); ax.set_ylabel("rain")
plt.show()

## Plate II · Learning doesn't help

Rosenblatt's rule never failed us in Lecture 2: when the neuron is wrong on a day, push its line towards that day. Here
every fix breaks another day. After 1,000 passes through the diary it still makes 10 mistakes a pass, and never fewer than 9.

In [ ]:
w, b, mistakes = np.zeros(2), 0.0, []
for _ in range(1000):
    m = 0
    for x, y in zip(X, Y):
        if (x @ w + b > 0) != (y > 0):
            s = 1 if y > 0 else -1
            w, b, m = w + s * x, b + s, m + 1
    mistakes.append(m)
print("mistakes in the last ten passes:", mistakes[-10:])
check("mistakes a pass, after 1,000 passes", mistakes[-1], 10)
check("fewest mistakes in any pass", min(mistakes), 9)

fig, ax = plt.subplots(figsize=(6, 2.6))
ax.plot(range(1, 61), mistakes[:60], "-o", color=INK, ms=3, lw=1)
ax.set_xlabel("pass through the diary"); ax.set_ylabel("mistakes"); ax.set_ylim(0, 12)
plt.show()

## Plate III · Two alarms

Give the job to two neurons, each Lecture 1's neuron drawing one line:

- **too cold**: total = 3 − 10 × warmth, so it fires when warmth is below 0.3;
- **too hot**: total = 10 × warmth − 7, so it fires when warmth is above 0.7.

Each one's gate shuts off the side where it is quiet.

In [ ]:
cold_fires = int(np.sum(3 - 10 * X[:, 0] > 0))
hot_fires = int(np.sum(10 * X[:, 0] - 7 > 0))
check("the cold alarm fires on", cold_fires, 12)
check("the hot alarm fires on", hot_fires, 11)

fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.axvspan(0, 0.3, facecolor="none", hatch="\\\\", edgecolor=INK2, lw=0)
ax.axvspan(0.7, 1, facecolor="none", hatch="\\\\", edgecolor=INK2, lw=0)
ax.axvline(0.3, color=INK, lw=2.5); ax.axvline(0.7, color=INK, lw=2.5)
ax.text(0.15, 1.02, "too cold", ha="center", style="italic"); ax.text(0.85, 1.02, "too hot", ha="center", style="italic")
draw_days(ax); ax.set_xlabel("warmth"); ax.set_ylabel("rain")
plt.show()

## Plate IV · The combiner

A third neuron, the **combiner**, listens to the two alarms. It starts from "go": its bias is 1. Each alarm votes against
it: −2 times whatever the alarm says. A vote is just a weight on the wire from an alarm. The combiner is Lecture 1's neuron
too, gate and all, so

  answer = max(0, 1 − 2 × cold − 2 × hot).

A **network** is Lecture 1's neuron, copied and wired together. Below, `Net` holds its numbers: `A` and `c` for the two
alarms (their weights and biases), `v` and `d` for the combiner (its votes and bias). `forward` runs a day through it.

In [ ]:
class Net:
    """2 inputs → hidden neurons (weights A, biases c) → one combiner (votes v, bias d); every neuron gated."""
    def __init__(self, A, c, v, d):
        self.A, self.c, self.v, self.d = np.array(A, float), np.array(c, float), np.array(v, float), float(d)

    def copy(self):
        return Net(self.A.copy(), self.c.copy(), self.v.copy(), self.d)

    def forward(self, X):
        z = X @ self.A.T + self.c          # each hidden neuron's total
        h = np.maximum(0.0, z)             # ... through its gate
        t = h @ self.v + self.d            # the combiner's total
        return z, h, t, np.maximum(0.0, t) # ... through its gate: the answer

    def right(self, X=X, Y=Y):
        return int(np.sum((self.forward(X)[3] > 0.5) == (Y > 0.5)))

    def error(self, X=X, Y=Y):
        return float(np.mean((self.forward(X)[3] - Y) ** 2))

hand = Net(A=[[-10, 0], [10, 0]], c=[3, -7], v=[-2, -2], d=1)     # our design, from words alone
z, h, t, out = hand.forward(np.array([[0.47, 0.26]]))
print(f"a day in the band (0.47, 0.26): cold {h[0, 0]:.1f}, hot {h[0, 1]:.1f}, answer {out[0]:.1f}: go")
check("a day in the band: answer", out[0], 1.0)

In [ ]:
def draw_net(ax, net, day=None, title=None):
    """The 2–2–1 network as on the plates: wires as thick as their weight, hollow if negative; values on the neurons."""
    IN, HID, OUT = [(0, 3), (0, 1)], [(3, 3.4), (3, 0.6)], (6, 2)
    vals = net.forward(np.array([day]))[1:] if day is not None else None
    def wire(p, q, w):
        lw = 0.8 + 0.4 * min(abs(w), 10)
        if abs(w) < 1e-9:
            ax.plot(*zip(p, q), color=INK2, lw=0.8, ls=":")
            return
        ax.plot(*zip(p, q), color=INK, lw=lw, solid_capstyle="butt")
        if w < 0 and lw > 2:
            ax.plot(*zip(p, q), color=PAPER, lw=lw - 1.8, solid_capstyle="butt")
    for j, hp in enumerate(HID):
        for i, ip in enumerate(IN):
            wire(ip, hp, net.A[j, i])
        wire(hp, OUT, net.v[j])
    for i, (p, name) in enumerate(zip(IN, ("warmth", "rain"))):
        ax.add_patch(plt.Circle(p, 0.32, fc=INKMAP(day[i]) if day is not None else PAPER, ec=INK, zorder=3))
        ax.text(p[0] - 0.45, p[1], name + (f"\n{day[i]:.3f}" if day is not None else ""), ha="right", va="center", fontsize=9)
    for j, (p, name) in enumerate(zip(HID, ("too cold", "too hot"))):
        hj = vals[0][0, j] if vals is not None else 0
        ax.add_patch(plt.Circle(p, 0.42, fc=INKMAP(min(1, hj) * 0.6), ec=INK, zorder=3))
        ax.text(p[0], p[1] + 0.6, name, ha="center", style="italic", fontsize=9)
        if vals is not None:
            ax.text(p[0], p[1], f"{hj:.2f}", ha="center", va="center", fontsize=9, zorder=4)
    o = vals[2][0] if vals is not None else 0
    ax.add_patch(plt.Circle(OUT, 0.5, fc=INKMAP(min(1, o) * 0.6), ec=INK, zorder=3))
    ax.text(OUT[0], OUT[1] + 0.75, "combiner", ha="center", style="italic", fontsize=9)
    if vals is not None:
        ax.text(OUT[0], OUT[1], f"{o:.2f}", ha="center", va="center", fontsize=10, zorder=4)
    ax.set_xlim(-1.6, 7); ax.set_ylim(-0.2, 4.4); ax.set_aspect("equal"); ax.axis("off")
    if title: ax.set_title(title, fontsize=10)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, (name, day) in zip(axes, (("a day in the band", (0.47, 0.26)), ("a cold day", (0.17, 0.09)), ("a hot day", (0.88, 0.14)))):
    draw_net(ax, hand, day, title=name)
plt.show()

## Plate V · A band, carved

Where either alarm fires, the combiner shuts. A strip is left: **39 of 42** right. The three wrong days are rainy ones
inside the band; our words said nothing about rain, so our network doesn't either. Lean the hot alarm with rain (a weight
of 3 on rain) and heavy rain sets it off too: 42 of 42.

In [ ]:
def verdict_map(ax, net, title=None, lines=True):
    g = np.linspace(0, 1, 201); Wg, Rg = np.meshgrid(g, g)
    out = net.forward(np.c_[Wg.ravel(), Rg.ravel()])[3].reshape(Wg.shape)
    if (out > 0.5).any():
        ax.contourf(Wg, Rg, out, levels=[0.5, max(out.max(), 0.51)], colors="none", hatches=["////"])
    if lines:
        for j in range(len(net.c)):
            Z = net.A[j, 0] * Wg + net.A[j, 1] * Rg + net.c[j]
            if Z.min() < 0 < Z.max():
                ax.contour(Wg, Rg, Z, levels=[0], colors=INK, linewidths=1.6, linestyles="-" if net.v[j] < 0 else "--")
    o = net.forward(X)[3]
    draw_days(ax, wrong=[i for i in range(42) if (o[i] > 0.5) != (Y[i] > 0.5)])
    if title: ax.set_title(title, fontsize=10)

lean = Net(A=[[-10, 0], [10, 3]], c=[3, -7], v=[-2, -2], d=1)
check("our two alarms: days right", hand.right(), 39)
check("the hot alarm leaning with rain: days right", lean.right(), 42)
wrong = [tuple(np.round(X[i], 4)) for i in range(42) if (hand.forward(X)[3][i] > 0.5) != (Y[i] > 0.5)]
print("the three wrong days (warmth, rain):", wrong)
check("the wrong days are all rainy (rain above 0.55)", all(r > 0.55 for _, r in wrong), True)

fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.2))
verdict_map(axes[0], hand, "our two alarms: 39 of 42")
verdict_map(axes[1], lean, "the hot alarm leans with rain: 42")
plt.show()

## Plates VI–VII · Layers, and who is to blame?

Neurons side by side form a **layer**. The middle layer is called **hidden**: nobody tells those neurons what to say. We
just did, by hand. A real network has to find out for itself, which brings back Lecture 2's question with a twist: in
Lecture 2 the neuron's own answer was graded, so it knew how wrong it was. A hidden neuron's answer isn't graded. We need
to pass the blame back.

## Plate VIII · Day A, forwards

A hot, wet day we did not enjoy: warmth 0.735, rain 0.824.

In [ ]:
day_A = np.array([0.735, 0.824])
z, h, t, out = hand.forward(day_A[None])
print(f"cold alarm 3 − 7.35 = {z[0, 0]:.2f}: shut, says {h[0, 0]:.0f}")
print(f"hot alarm 7.35 − 7 = {z[0, 1]:.2f}: says {h[0, 1]:.2f}")
print(f"combiner 1 − 2 × 0 − 2 × {h[0, 1]:.2f} = {t[0]:.2f}: answer {out[0]:.2f}; error ({out[0]:.1f} − 0)² = {out[0] ** 2:.2f}")
check("cold alarm's total", z[0, 0], -4.35)
check("hot alarm says", h[0, 1], 0.35)
check("the answer", out[0], 0.3)
check("the error", out[0] ** 2, 0.09)
draw_net(plt.subplots(figsize=(5, 3))[1], hand, tuple(day_A), title="day A, forwards"); plt.show()

## Plate IX · Blame flows back

Run it backwards. **Blame** here means the slope of the error, carried backwards one wire at a time:

1. On the answer: twice the miss, 2 × 0.3 = **0.6** (Lecture 2's first factor). The combiner's gate is open, so all of it
   gets in.
2. Along each vote, times that vote's weight (−2): **−1.2** reaches each alarm. The combiner's own weights get the blame
   times what came in on them.
3. Through each alarm's gate: the cold alarm's is shut, so its blame stops dead (**0**); the hot alarm's is open.
4. Into the hot alarm's weights, times the input each one saw.

This is **backpropagation**: Lecture 2's three-factor chain, once per wire, from the end backwards.

In [ ]:
def backprop(net, day, target):
    """Blame on one day, written out wire by wire. Returns every slope and the blame at each stage."""
    z, h, t, out = (v[0] for v in net.forward(day[None]))
    on_answer = 2 * (out - target)                  # blame on the answer
    into_combiner = on_answer * (t > 0)             # ... through the combiner's gate
    slope_v, slope_d = into_combiner * h, into_combiner
    along_votes = into_combiner * net.v             # blame reaching each hidden neuron
    through_gates = along_votes * (z > 0)           # ... a shut gate passes nothing
    slope_A, slope_c = np.outer(through_gates, day), through_gates
    return dict(on_answer=on_answer, along_votes=along_votes, through_gates=through_gates,
                slope_A=slope_A, slope_c=slope_c, slope_v=slope_v, slope_d=slope_d)

bA = backprop(hand, day_A, 0.0)
check("blame on the answer", bA["on_answer"], 0.6)
check("blame along each vote", bA["along_votes"][1], -1.2)
check("through the cold alarm's shut gate", bA["through_gates"][0], 0.0)
check("through the hot alarm's open gate", bA["through_gates"][1], -1.2)
check("hot alarm, warmth weight: slope", bA["slope_A"][1, 0], -0.882)
check("hot alarm, rain weight: slope", bA["slope_A"][1, 1], -0.9888)
check("hot alarm, bias: slope", bA["slope_c"][1], -1.2)
check("the hot alarm's vote: slope", bA["slope_v"][1], 0.21)
check("the combiner's bias: slope", bA["slope_d"], 0.6)

Check the chain the way Lecture 2 did, by nudging: move one number a little up and a little down (0.001 each way) and see
how the error moves. Nudging both ways cancels the error's own curve, so the two agree to many decimals.

In [ ]:
def day_error(net, day, target):
    return (net.forward(day[None])[3][0] - target) ** 2

def nudge(part, idx, h=1e-3):
    up, down = hand.copy(), hand.copy()
    getattr(up, part)[idx] += h; getattr(down, part)[idx] -= h
    return (day_error(up, day_A, 0) - day_error(down, day_A, 0)) / (2 * h)

pairs = [(nudge("A", (1, 0)), bA["slope_A"][1, 0], "hot alarm, warmth weight"), (nudge("A", (1, 1)), bA["slope_A"][1, 1], "hot alarm, rain weight"),
         (nudge("c", 1), bA["slope_c"][1], "hot alarm, bias"), (nudge("v", 1), bA["slope_v"][1], "the hot alarm's vote"),
         (nudge("A", (0, 0)), bA["slope_A"][0, 0], "cold alarm, warmth weight")]
for by_nudge, by_chain, name in pairs:
    print(f"{name:>26}: nudge {by_nudge:+.4f}   chain {by_chain:+.4f}")
check("largest gap between nudge and chain", max(abs(a - b) for a, b, _ in pairs), 0.0, tol=1e-6)

## Plate X · The hot alarm learns about rain

Step every number against its blame, step size 0.1. Look at the hot alarm's weight on rain: it was 0 (our words never
mentioned rain). Now it is +0.099. Run day A again: the hot alarm grows louder, the combiner's total falls below zero, its
gate shuts, and the answer is 0.

In [ ]:
def step(net, g, lr):
    n = net.copy()
    n.A -= lr * g["slope_A"]; n.c -= lr * g["slope_c"]; n.v -= lr * g["slope_v"]; n.d -= lr * g["slope_d"]
    return n

after = step(hand, bA, 0.1)
z2, h2, t2, out2 = (v[0] for v in after.forward(day_A[None]))
print(f"hot alarm: warmth {after.A[1, 0]:.4f}, rain {after.A[1, 1]:.4f}, bias {after.c[1]:.2f}; votes {after.v}, combiner bias {after.d:.2f}")
print(f"day A again: hot alarm {h2[1]:.3f}, combiner total {t2:.4f}, answer {out2:.1f}")
check("hot alarm's new rain weight", after.A[1, 1], 0.0989, tol=5e-5)
check("hot alarm's new warmth weight", after.A[1, 0], 10.0882, tol=5e-5)
check("combiner's new bias", after.d, 0.94)
check("combiner's total on day A, after the step", t2, -0.3056, tol=5e-5)
check("the answer on day A, after the step", out2, 0.0)

## Plate XI · A day nobody hears

Day B: a rainy day inside the band (warmth 0.616, rain 0.727), not enjoyed. Both alarms are silent, so the combiner says 1:
the error is 1, as big as it gets. Blame 2 flows back along both votes (−4 to each alarm) and meets two shut gates. Not
one hidden weight hears about this day. Only the combiner's bias can move (1 → 0.8).

In [ ]:
day_B = np.array([0.616, 0.727])
bB = backprop(hand, day_B, 0.0)
print("blame along the votes:", bB["along_votes"], " through the gates:", bB["through_gates"])
check("day B: blame on the answer", bB["on_answer"], 2.0)
check("day B: blame along each vote", bB["along_votes"][0], -4.0)
check("day B: largest slope on any hidden weight", float(np.abs(bB["slope_A"]).max()), 0.0)
check("day B: the combiner's bias after one step", step(hand, bB, 0.1).d, 0.8)

(Plate XII is the narrated film of these two days.)

## Plate XIII · Stuck at 39

Let our hand-built network learn from all 42 days at once, 20,000 steps of size 0.1, the blame averaged over the days.
Nothing happens: it stays at 39 of 42. The days it gets wrong reach neither alarm (day B's story). The hot, wet days that
could teach the hot alarm about rain are ones the combiner already rejects firmly: its own gate is shut on them, so their
blame stops at the combiner.

In [ ]:
def train(net, lr, steps, record_every=0):
    """Full-batch learning: the average of every day's blame, one step at a time."""
    n, hist = net.copy(), []
    for k in range(steps + 1):
        if record_every and k % record_every == 0:
            hist.append((k, n.right(), n.d, n.A[1, 1]))
        if k == steps:
            break
        z, h, t, out = n.forward(X)
        into = 2 * (out - Y) / len(X) * (t > 0)
        through = np.outer(into, n.v) * (z > 0)
        n.A -= lr * through.T @ X; n.c -= lr * through.sum(0)
        n.v -= lr * h.T @ into;    n.d -= lr * into.sum()
    return n, hist

stuck, hist = train(hand, 0.1, 20000, record_every=1000)
print("days right every 1,000 steps:", [r for _, r, _, _ in hist])
print(f"the combiner's bias: 1 → {stuck.d:.4f}; the hot alarm's rain weight: 0 → {stuck.A[1, 1]:.4f}")
check("days right after 20,000 steps", stuck.right(), 39)
check("days right never changes", len({r for _, r, _, _ in hist}), 1)
check("the combiner's bias settles at", stuck.d, 0.8421, tol=5e-5)
check("the hot alarm's rain weight creeps to", stuck.A[1, 1], 0.0266, tol=5e-5)

## Plate XIV · Twelve random starts

Start from random numbers instead. We train many copies of a network with two hidden neurons side by side, the way the
lecture did, one random start (*seed*) per copy:

- each hidden neuron's two weights are drawn at random (He's rule: normal, with spread √(2 ÷ inputs));
- each hidden line starts through the middle of the diary, (0.5, 0.5);
- the combiner's bias starts at 0.5, so at first it fires on every day;
- 20,000 steps of size 0.1.

`Nets` is `Net` for many seeds at once (the same arithmetic, an extra first axis).

In [ ]:
class Nets:
    """Many copies of the network, one per random seed, trained side by side."""
    def __init__(self, seeds, H, d0=0.5):
        A, v = [], []
        for s in seeds:
            rng = np.random.default_rng(s)
            A.append(rng.normal(0, np.sqrt(2 / 2), (H, 2)))
            v.append(rng.normal(0, np.sqrt(2 / H), H))
        self.A, self.v = np.array(A), np.array(v)
        self.c = -self.A @ np.array([0.5, 0.5])         # every hidden line through the middle of the diary
        self.d = np.full(len(seeds), float(d0))

    def forward(self, X):
        z = np.einsum("nk,shk->snh", X, self.A) + self.c[:, None, :]
        h = np.maximum(0.0, z)
        t = np.einsum("snh,sh->sn", h, self.v) + self.d[:, None]
        return z, h, t, np.maximum(0.0, t)

    def right(self, X=X, Y=Y):
        return ((self.forward(X)[3] > 0.5) == (Y > 0.5)).sum(axis=1)

    def train(self, lr=0.1, steps=20000):
        for _ in range(steps):
            z, h, t, out = self.forward(X)
            b_t = 2 * (out - Y) / len(X) * (t > 0)
            g_v = np.einsum("snh,sn->sh", h, b_t)
            g_d = b_t.sum(1)
            b_z = b_t[:, :, None] * self.v[:, None, :] * (z > 0)
            g_A = np.einsum("snh,nk->shk", b_z, X)
            self.A -= lr * g_A; self.c -= lr * b_z.sum(1); self.v -= lr * g_v; self.d -= lr * g_d
        return self

    def net(self, s):
        return Net(self.A[s], self.c[s], self.v[s], self.d[s])

In [ ]:
import time
t0 = time.time()                                                     # about three minutes
runs = {H: Nets(range(300), H).train() for H in (2, 3, 4, 8)}          # 300 random starts at each width
print(f"trained 1,200 networks in {time.time() - t0:.0f} s")
two = runs[2]
right2 = two.right()
print("the first twelve starts, days right:", right2[:12].tolist())
check("of the first twelve, all 42 right", int(np.sum(right2[:12] == 42)), 5)
check("of 300 starts, all 42 right", int(np.sum(right2 == 42)), 126, tol=3)

fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for s, ax in enumerate(axes.ravel()):
    verdict_map(ax, two.net(s))
    ax.set_title(f"start {s}: {right2[s]} of 42", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

Glorot, Bordes and Bengio (2011) found the gate's hard zero does little harm while some units in each layer stay on. Our
combiner is a layer of one. The lecture's margin note: started at bias 0 instead of 0.5, it falls silent on every day in
45–57% of 300 runs (2–8 hidden neurons); started at 0.5, in 0–6%. A combiner silent on every day says *stay in* to all
42 days and gets exactly the 26 days we did not enjoy right. We retrain from bias 0 to count those.

In [ ]:
t0 = time.time()                                                     # about three minutes
zero = {H: Nets(range(300), H, d0=0.0).train() for H in (2, 3, 4, 8)}
print(f"trained 1,200 more networks in {time.time() - t0:.0f} s")
silent0 = {H: float(np.mean(n.right() == 26)) for H, n in zero.items()}
silent5 = {H: float(np.mean(n.right() == 26)) for H, n in runs.items()}
for H in (2, 3, 4, 8):
    print(f"{H} hidden neurons: silent combiner {silent0[H]:.0%} from bias 0, {silent5[H]:.0%} from bias 0.5")
check("from bias 0, the fewest silent (of the four widths)", min(silent0.values()), 0.447, tol=0.02)
check("from bias 0, the most silent", max(silent0.values()), 0.567, tol=0.02)
check("from bias 0.5, the most silent", max(silent5.values()), 0.057, tol=0.02)

## Plate XV · Three ways to carve a band

Read each successful two-neuron network by its two hidden neurons: which side of its line each fires on (the **cold side**
if its weight on warmth is negative, the **warm side** if positive) and how it votes (**for** or **against**). Every success
is one of exactly three designs:

- **two alarms** (cold side, against + warm side, against): our design, improved;
- **a bump from the warm side**: one votes for warmth, the other against too much of it;
- **a bump from the cold side**: the mirror image.

And which neuron holds which job is an accident: in the two-alarm design the cold alarm is neuron 1 in about half the runs.

In [ ]:
from collections import Counter

def describe(net):
    z = X @ net.A.T + net.c
    parts = []
    for j in range(len(net.c)):
        if not np.any(z[:, j] > 0):
            parts.append(("dead", j))
        else:
            parts.append((("cold side" if net.A[j, 0] < 0 else "warm side") + "/" + ("for" if net.v[j] > 0 else "against"), j))
    return " + ".join(sorted(p for p, _ in parts)), parts

wins = [s for s in range(300) if right2[s] == 42]
families = Counter(describe(two.net(s))[0] for s in wins)
for k, n in families.most_common():
    print(f"{n:3d}  {k}")
alarms = "cold side/against + warm side/against"
cold_in_first = sum(1 for s in wins if describe(two.net(s))[0] == alarms and describe(two.net(s))[1][0][0] == "cold side/against")
check("number of designs", len(families), 3)
check("two alarms", families[alarms], 60, tol=2)
check("a bump from the warm side", families["warm side/against + warm side/for"], 39, tol=2)
check("a bump from the cold side", families["cold side/against + cold side/for"], 27, tol=2)
check("two alarms with the cold alarm in neuron 0", cold_in_first, 28, tol=2)

NAMES = {alarms: "two alarms", "warm side/against + warm side/for": "a bump from the warm side",
         "cold side/against + cold side/for": "a bump from the cold side"}
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, (k, n) in zip(axes, families.most_common()):
    s = min((s for s in wins if describe(two.net(s))[0] == k), key=lambda s: two.net(s).error())
    verdict_map(ax, two.net(s), title=f"{NAMES[k]}: {n} runs\n(the best, start {s}; dashed: votes for)")
plt.tight_layout(); plt.show()

## Plate XVI · Spare neurons

Rumelhart, Hinton and Williams (1986) noted that a network with two hidden units *sometimes* sticks short of solving XOR.
On our band diary two neurons stick most of the time: 174 of 300 starts never get all 42 right, most often stopping at 27.
With spare neurons fewer get stuck; with eight, all 300 find the band.

In [ ]:
wins_by_width = {H: int(np.sum(n.right() == 42)) for H, n in runs.items()}
hist2 = Counter(right2.tolist())
print("two neurons, days right over 300 starts:", dict(sorted(hist2.items())))
print("all 42 right, of 300, by width:", wins_by_width)
check("two neurons: starts stuck short of 42", 300 - wins_by_width[2], 174, tol=3)
check("two neurons: the commonest place to stop", max((n, r) for r, n in hist2.items() if r < 42)[1], 27)
check("three neurons: all 42 right", wins_by_width[3], 218, tol=3)
check("four neurons: all 42 right", wins_by_width[4], 267, tol=3)
check("eight neurons: all 42 right", wins_by_width[8], 300)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
rs = sorted(hist2)
axes[0].bar(rs, [hist2[r] for r in rs], color=[INK if r == 42 else PAPER for r in rs], edgecolor=INK, hatch=None)
axes[0].set_xlabel("days right"); axes[0].set_ylabel("starts"); axes[0].set_title("two neurons, 300 starts", fontsize=10)
axes[1].bar([str(H) for H in wins_by_width], list(wins_by_width.values()), color=INK)
axes[1].set_xlabel("hidden neurons"); axes[1].set_ylabel("all 42 right (of 300)")
plt.tight_layout(); plt.show()

## Plates XVII–XVIII · History, and check yourself

Minsky and Papert (1969) proved what a single layer cannot compute. The idea of passing blame backwards arrived several
times between 1970 and 1985, and Rumelhart, Hinton and Williams (1986) showed it trains hidden units that come to stand
for features of the task. They looked inside to check. So will we, in Lecture 4.

### Your turn 1 · A day with warmth 0.5 and rain 0.2

What does each alarm say, and what does the network answer?

*Hint:* `hand.forward(np.array([[0.5, 0.2]]))` returns the hidden totals, the hidden outputs, the combiner's total and
the answer.

In [ ]:
answer_mild = None   # TODO

### Your turn 2 · A silent alarm hears nothing

On a day when the hot alarm is silent, how much blame reaches its weights? Try the mild day above with target 0.

*Hint:* `backprop(hand, day, target)["slope_A"][1]` holds the hot alarm's two slopes.

In [ ]:
hot_slopes_mild = None   # TODO

### Your turn 3 · How much lean?

The hot alarm leaning with rain weight 3 gets all 42 right. What is the smallest rain weight (in steps of 0.1) that still
gets all 42?

*Hint:* loop over `np.arange(0, 3.01, 0.1)` and build `Net(A=[[-10, 0], [10, lean]], ...)`.

In [ ]:
smallest_lean = None   # TODO

### Your turn 4 · A smaller step

On Plate X, slide the step size from 0.1 down to 0.02. What does day A's answer become after one step?

*Hint:* `step(hand, bA, 0.02)`, then run day A forwards.

In [ ]:
answer_small_step = None   # TODO

### Your turn 5 · One hidden neuron

With one hidden neuron, how many of 300 random starts get every day right?

*Hint:* `Nets(range(300), 1).train()`. Think before you run it: what shape can one neuron and a combiner carve?

In [ ]:
wins_one_neuron = None   # TODO

## What we built

- One neuron draws one line; some patterns need several.
- Hidden neurons feed a combiner; together they carve shapes no line can.
- Backpropagation: blame starts at the answer and flows back along every wire, times its weight, through open gates only.
- Learning finds its own version of the jobs we would have given the neurons, sometimes a better one, and not always in
  the same neuron.

**Next · Lecture 4, Reading a Small Network:** we built this network, so we know what each neuron does. What about one we
didn't build?

---

## Solutions

**Your turn 1.** Cold: 3 − 5 < 0, silent. Hot: 5 − 7 < 0, silent. The combiner says 1: go.

In [ ]:
z, h, t, out = hand.forward(np.array([[0.5, 0.2]]))
answer_mild = out[0]
check("both alarms silent", float(h.max()), 0.0)
check("the answer", answer_mild, 1.0)

**Your turn 2.** None: a shut gate passes nothing back.

In [ ]:
hot_slopes_mild = backprop(hand, np.array([0.5, 0.2]), 0.0)["slope_A"][1]
check("blame reaching the silent hot alarm", float(np.abs(hot_slopes_mild).max()), 0.0)

**Your turn 3.** It takes almost the whole lean: 2.9. On each of the three rainy days the hot alarm must say more than
0.25, so that the combiner's 1 − 2 × hot falls below ½; the least warm of them (warmth 0.537, rain 0.667) needs a rain
weight of at least (7.25 − 5.37) ÷ 0.667 = 2.82.

In [ ]:
smallest_lean = next(round(l, 1) for l in np.arange(0, 3.01, 0.1) if Net(A=[[-10, 0], [10, l]], c=[3, -7], v=[-2, -2], d=1).right() == 42)
print("the smallest lean that gets all 42 right:", smallest_lean)
check("leaning that much gets all 42", Net(A=[[-10, 0], [10, smallest_lean]], c=[3, -7], v=[-2, -2], d=1).right(), 42)
check("the smallest lean, in steps of 0.1", smallest_lean, 2.9)

**Your turn 4.** The answer falls from 0.3 to 0.18: a smaller step moves less.

In [ ]:
answer_small_step = float(step(hand, bA, 0.02).forward(day_A[None])[3][0])
check("day A's answer after a step of 0.02", answer_small_step, 0.1798, tol=5e-5)

**Your turn 5.** None: one neuron draws one line, and a combiner listening to one line can still only say yes on one side of it.

In [ ]:
wins_one_neuron = int(np.sum(Nets(range(300), 1).train().right() == 42))
check("one hidden neuron: all 42 right", wins_one_neuron, 0)